# Módulo 01 · Aula 04 — Funções e Módulos

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Até aqui seu código é um bloco só. Se a regra de frete muda, você caça a linha em cinco lugares. Funções resolvem isso.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Definir e chamar funções | Reuso e nomeação de ideias |
| 2 | Parâmetros e argumentos | Posicionais, nomeados, padrão |
| 3 | `return` | Devolver resultados (e não só imprimir) |
| 4 | Docstrings e type hints | Código que se documenta |
| 5 | Escopo LEGB | Onde cada nome vive |
| 6 | Armadilha do argumento mutável | Bug clássico de entrevista |
| 7 | Módulos e `import` | Organizar código em arquivos |
| 8 | `if __name__ == "__main__"` | Script vs biblioteca |

## 1. Definindo funções

```python
def nome_da_funcao(parametro1, parametro2):
    """Docstring: o que a função faz."""
    # corpo
    return resultado
```

**Três razões para criar uma função:**

1. **Reuso** — escreva uma vez, use em muitos lugares.
2. **Nomeação** — `calcular_frete(...)` explica a intenção melhor que 8 linhas soltas.
3. **Testabilidade** — você consegue verificar uma função isoladamente (Módulo 12).

**Regra prática:** se você copiou e colou um trecho, ele deveria ser uma função.

In [ ]:
def saudacao(nome):
    """Devolve uma saudação personalizada."""
    return f"Olá, {nome}! Bem-vindo ao Atlas."


# Definir não executa. Chamar executa.
print(saudacao("Maria"))
print(saudacao("Carlos"))

In [ ]:
def calcular_frete(valor_pedido, cidade):
    """Aplica a regra de frete da Aurora Comércio."""
    if valor_pedido >= 500:
        return 0.0
    if cidade == "Campinas":
        return 9.90
    return 29.90


for v, c in [(600, "São Paulo"), (150, "Campinas"), (150, "Santos")]:
    print(f"R$ {v:>6,.2f} para {c:<12} -> frete R$ {calcular_frete(v, c):.2f}")

## 2. Parâmetros e argumentos

- **Parâmetro** = o nome na definição.
- **Argumento** = o valor passado na chamada.

**Formas de passar argumentos:**

| Forma | Exemplo | Observação |
|-------|---------|------------|
| Posicional | `f(10, "SP")` | A ordem importa |
| Nomeado (*keyword*) | `f(cidade="SP", valor=10)` | Ordem livre, muito mais legível |
| Com valor padrão | `def f(x, taxa=0.1)` | Padrões vêm **depois** dos obrigatórios |

In [ ]:
def registrar_venda(produto, quantidade, preco, desconto=0.0, cidade="Campinas"):
    """Calcula o total de uma venda com desconto opcional."""
    bruto = quantidade * preco
    liquido = bruto * (1 - desconto)
    return {
        "produto": produto,
        "cidade": cidade,
        "bruto": round(bruto, 2),
        "desconto": round(bruto - liquido, 2),
        "liquido": round(liquido, 2),
    }


print(registrar_venda("Notebook", 2, 2599.90))
print(registrar_venda("Mouse", 10, 89.90, 0.15))
print(registrar_venda("Monitor", 1, 1199.00, cidade="Sorocaba", desconto=0.05))

### Argumentos nomeados aumentam MUITO a legibilidade

Compare:

```python
processar(dados, True, False, 3)          # o que significa cada um?
processar(dados, validar=True, verbose=False, tentativas=3)   # óbvio
```

Regra prática: se um parâmetro é booleano ou um número mágico, passe nomeado.

In [ ]:
# Forçando argumentos nomeados com *
def gerar_relatorio(dados, *, formato="texto", incluir_total=True, ordenar_por="valor"):
    """Os parâmetros após o * SÓ podem ser passados por nome."""
    return f"Relatório [{formato}] ordenado por {ordenar_por} | total={incluir_total} | {len(dados)} registros"


print(gerar_relatorio([1, 2, 3], formato="csv", ordenar_por="cidade"))

# gerar_relatorio([1,2,3], "csv")   # <- descomente: TypeError

## 3. `return` — devolver em vez de imprimir

Esta é uma das distinções mais importantes para quem está começando.

| `print()` | `return` |
|-----------|----------|
| Mostra na tela | Entrega o valor para quem chamou |
| Devolve `None` | Devolve o objeto |
| Serve ao humano | Serve ao programa |

**Uma função que só imprime é um beco sem saída** — você não consegue usar o resultado em outro cálculo. Funções de cálculo devem **retornar**; a impressão fica a cargo de quem chamou.

`return` também **encerra** a função imediatamente. Múltiplos `return` são perfeitamente idiomáticos (guard clauses).

In [ ]:
def soma_ruim(a, b):
    print(a + b)          # só mostra


def soma_boa(a, b):
    return a + b          # entrega


x = soma_ruim(2, 3)
y = soma_boa(2, 3)

print("\nx =", x, "(inútil para cálculos)")
print("y =", y)
print("Posso continuar:", y * 10)

In [ ]:
# Múltiplos returns como "guard clauses" — evita aninhamento profundo
def classificar_pedido(valor, status):
    """Classifica um pedido. Retorna cedo nos casos triviais."""
    if status == "cancelado":
        return "Ignorado"
    if status != "pago":
        return "Aguardando"
    if valor >= 5000:
        return "Alto valor"
    if valor >= 1000:
        return "Médio valor"
    return "Baixo valor"


casos = [(6000, "pago"), (1500, "pago"), (200, "pago"), (900, "pendente"), (300, "cancelado")]
for valor, status in casos:
    print(f"R$ {valor:>6,} {status:<11} -> {classificar_pedido(valor, status)}")

In [ ]:
# Retornando múltiplos valores (na verdade, uma tupla)
def estatisticas(valores):
    """Devolve total, média, mínimo e máximo."""
    return sum(valores), sum(valores) / len(valores), min(valores), max(valores)


vendas = [45300.00, 128900.50, 22100.00, 67450.75]

total, media, minimo, maximo = estatisticas(vendas)   # desempacotamento!
print(f"Total:  R$ {total:>12,.2f}")
print(f"Média:  R$ {media:>12,.2f}")
print(f"Mínimo: R$ {minimo:>12,.2f}")
print(f"Máximo: R$ {maximo:>12,.2f}")

print("\nO que realmente volta:", type(estatisticas(vendas)).__name__)

### Função sem `return` devolve `None`

Se você esquecer o `return`, a função devolve `None` silenciosamente. É a origem de muitos `TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'`.

In [ ]:
def esqueci_o_return(a, b):
    resultado = a + b


r = esqueci_o_return(2, 3)
print("r =", r, "| tipo:", type(r).__name__)
# print(r * 10)   # <- descomente: TypeError

## 4. Docstrings e type hints

**Docstring** — string logo abaixo do `def`, acessível via `help(funcao)` ou `funcao.__doc__`. É a documentação oficial da função.

**Type hints** — anotações de tipo (`valor: float -> str`). Python **não** as verifica em tempo de execução (não impedem você de passar o tipo errado), mas:

- o VS Code usa para autocompletar e apontar erros;
- ferramentas como `mypy` verificam estaticamente;
- servem de documentação executável.

No Módulo 04 aprofundamos em tipagem; aqui basta pegar o hábito.

In [ ]:
def calcular_desconto(valor: float, percentual: float = 0.0, teto: float | None = None) -> float:
    """Calcula o valor do desconto de um pedido.

    Args:
        valor: Valor bruto do pedido em reais.
        percentual: Percentual de desconto (0.1 = 10%). Padrão 0.
        teto: Valor máximo de desconto em reais. None = sem limite.

    Returns:
        O valor do desconto, em reais, já respeitando o teto.

    Examples:
        >>> calcular_desconto(1000.0, 0.10)
        100.0
        >>> calcular_desconto(1000.0, 0.10, teto=50.0)
        50.0
    """
    desconto = valor * percentual
    if teto is not None:
        desconto = min(desconto, teto)
    return round(desconto, 2)


print(calcular_desconto(1000.0, 0.10))
print(calcular_desconto(1000.0, 0.10, teto=50.0))
print()
help(calcular_desconto)

## 5. Escopo — a regra LEGB

Quando Python encontra um nome, ele procura nesta ordem:

```
L — Local      : dentro da função atual
E — Enclosing  : na função que envolve esta (funções aninhadas)
G — Global     : no nível do módulo/notebook
B — Built-in   : nomes embutidos (print, len, sum...)
```

Achou, para. Não achou em nenhum: `NameError`.

**Regra fundamental:** dentro de uma função você pode **ler** uma variável global, mas **atribuir** a ela cria uma variável **local** nova — a global fica intacta.

In [ ]:
TAXA_IMPOSTO = 0.18          # Global


def aplicar_imposto(valor):
    ajuste = 1.0             # Local
    return valor * (1 + TAXA_IMPOSTO) * ajuste     # lê a global sem problema


print(aplicar_imposto(1000))
# print(ajuste)   # <- descomente: NameError, 'ajuste' só existe dentro da função

In [ ]:
contador = 0        # global


def incrementa_errado():
    contador = 99   # cria uma variável LOCAL nova, não toca a global
    return contador


print("dentro :", incrementa_errado())
print("fora   :", contador, "<- global intacta")

In [ ]:
# global e nonlocal: use com MUITA parcimônia
contador = 0


def incrementa_certo():
    global contador
    contador += 1


incrementa_certo()
incrementa_certo()
print("contador global:", contador)


# nonlocal: alcança o escopo Enclosing
def criar_acumulador():
    total = 0                      # escopo Enclosing

    def somar(valor):
        nonlocal total             # sem isso, 'total' viraria local de somar()
        total += valor
        return total

    return somar


acc = criar_acumulador()
print("\nacumulando:", acc(100), acc(50), acc(25))

> ⚠️ **`global` é quase sempre um cheiro de código ruim.** Ele cria acoplamento invisível: qualquer função pode alterar o estado de qualquer outra. Prefira **passar por parâmetro e retornar**. Funções que dependem só das suas entradas (funções *puras*) são triviais de testar e raciocinar.

## 6. ⚠️ A armadilha do argumento padrão mutável

**O bug clássico.** O valor padrão de um parâmetro é avaliado **uma única vez**, quando a função é definida — não a cada chamada. Se o padrão for mutável (`[]`, `{}`, `set()`), todas as chamadas compartilham o **mesmo objeto**.

In [ ]:
def adicionar_item_ERRADO(item, carrinho=[]):
    carrinho.append(item)
    return carrinho


print(adicionar_item_ERRADO("Mouse"))
print(adicionar_item_ERRADO("Teclado"))     # 😱 o Mouse ainda está lá
print(adicionar_item_ERRADO("Monitor"))

In [ ]:
# A CORREÇÃO: use None como sentinela
def adicionar_item_CERTO(item, carrinho=None):
    if carrinho is None:
        carrinho = []          # lista NOVA a cada chamada
    carrinho.append(item)
    return carrinho


print(adicionar_item_CERTO("Mouse"))
print(adicionar_item_CERTO("Teclado"))
print(adicionar_item_CERTO("Monitor", ["Notebook"]))   # ainda aceita uma lista existente

### Mutável vs imutável em argumentos

Python passa argumentos por **atribuição de nome** ("*pass by object reference*"). Consequência prática:

- Se você **reatribui** o parâmetro dentro da função, o chamador não vê nada.
- Se você **muta** o objeto (append, `d[k]=v`), o chamador **vê**.

In [ ]:
def reatribui(lista):
    lista = ["novo"]        # só troca o rótulo local
    return lista


def muta(lista):
    lista.append("novo")    # altera o objeto do chamador!
    return lista


a = ["original"]
reatribui(a)
print("depois de reatribui:", a)

b = ["original"]
muta(b)
print("depois de muta:    ", b)

# Se a função não deveria alterar a entrada, copie:
def muta_seguro(lista):
    lista = lista.copy()
    lista.append("novo")
    return lista


c = ["original"]
resultado = muta_seguro(c)
print("\nc         :", c)
print("resultado :", resultado)

## 7. Módulos e `import`

Um **módulo** é simplesmente um arquivo `.py`. Um **pacote** é uma pasta com módulos.

### Formas de importar

```python
import math                     # importa o módulo inteiro -> math.sqrt(x)
import statistics as stats      # com apelido            -> stats.mean(x)
from math import sqrt, pi       # importa nomes soltos    -> sqrt(x)
from math import sqrt as raiz   # nome solto com apelido
from math import *              # ❌ NUNCA. Polui o namespace e esconde a origem.
```

**Convenção PEP 8** para a ordem dos imports, separados por linha em branco:

1. Biblioteca padrão (`os`, `json`, `datetime`)
2. Bibliotecas de terceiros (`pandas`, `requests`)
3. Módulos do seu próprio projeto

In [ ]:
import math
import statistics as stats
from datetime import date, timedelta

valores = [45300.00, 128900.50, 22100.00, 67450.75, 91200.00]

print("Raiz de 144      :", math.sqrt(144))
print("Teto de 4.2      :", math.ceil(4.2))
print("Piso de 4.8      :", math.floor(4.8))
print()
print("Média            :", f"R$ {stats.mean(valores):,.2f}")
print("Mediana          :", f"R$ {stats.median(valores):,.2f}")
print("Desvio padrão    :", f"R$ {stats.stdev(valores):,.2f}")
print()
hoje = date(2026, 8, 12)
print("Hoje             :", hoje.strftime("%d/%m/%Y"))
print("Daqui a 30 dias  :", (hoje + timedelta(days=30)).strftime("%d/%m/%Y"))

### Módulos da biblioteca padrão que você vai usar sempre

| Módulo | Para quê |
|--------|----------|
| `os`, `pathlib` | Caminhos e sistema de arquivos |
| `json` | Ler/escrever JSON |
| `csv` | Ler/escrever CSV |
| `datetime` | Datas e horas |
| `math`, `statistics` | Matemática e estatística |
| `random` | Aleatoriedade (dados de teste!) |
| `collections` | `defaultdict`, `Counter`, `namedtuple` |
| `itertools` | Combinatória e iteração avançada |
| `re` | Expressões regulares |
| `logging` | Logs estruturados (Módulo 04) |

In [ ]:
import random

random.seed(42)     # semente fixa = resultados reprodutíveis. SEMPRE em testes.

cidades = ["Campinas", "São Paulo", "Sorocaba", "Ribeirão Preto"]
produtos = ["Notebook", "Mouse", "Teclado", "Monitor"]

print("Gerando 5 pedidos sintéticos:\n")
for i in range(1, 6):
    pedido = {
        "id": 2000 + i,
        "cidade": random.choice(cidades),
        "produto": random.choice(produtos),
        "qtd": random.randint(1, 10),
        "preco": round(random.uniform(50, 3000), 2),
    }
    print(pedido)

## 8. Criando seu próprio módulo

Vamos escrever um arquivo `.py` de verdade a partir do notebook e depois importá-lo. É exatamente o que você fará no `projeto_Atlas`.

O comando mágico `%%writefile` grava o conteúdo da célula em um arquivo.

In [ ]:
%%writefile atlas_utils.py
"""Utilitários de cálculo da Aurora Comércio.

Este módulo é importável e também executável diretamente.
"""

TAXA_IMPOSTO = 0.18
FRETE_GRATIS_ACIMA_DE = 500.0


def calcular_frete(valor_pedido: float, cidade: str) -> float:
    """Aplica a regra de frete da Aurora."""
    if valor_pedido >= FRETE_GRATIS_ACIMA_DE:
        return 0.0
    return 9.90 if cidade == "Campinas" else 29.90


def calcular_total(quantidade: int, preco: float, desconto: float = 0.0) -> float:
    """Total líquido de um item, com desconto percentual."""
    return round(quantidade * preco * (1 - desconto), 2)


def formatar_brl(valor: float) -> str:
    """Formata um número no padrão monetário brasileiro."""
    inteiro = f"{valor:,.2f}"
    return "R$ " + inteiro.replace(",", "X").replace(".", ",").replace("X", ".")


def _uso_interno():
    """O _ no início sinaliza 'privado por convenção'."""
    return "não faz parte da API pública"


if __name__ == "__main__":
    # Este bloco SÓ roda quando o arquivo é executado diretamente.
    print("=== Teste rápido de atlas_utils ===")
    print("Frete SP  R$150 :", calcular_frete(150, "São Paulo"))
    print("Frete CPS R$150 :", calcular_frete(150, "Campinas"))
    print("Frete CPS R$600 :", calcular_frete(600, "Campinas"))
    print("Total 3x99.90   :", formatar_brl(calcular_total(3, 99.90)))

In [ ]:
# Agora importamos o arquivo que acabamos de criar
import atlas_utils

print(atlas_utils.calcular_frete(150, "Campinas"))
print(atlas_utils.formatar_brl(1234567.891))
print()
print("Constante:", atlas_utils.TAXA_IMPOSTO)
print("API pública:", [n for n in dir(atlas_utils) if not n.startswith("_")])

In [ ]:
# Executando o arquivo como SCRIPT — agora o bloco __main__ roda
!python atlas_utils.py

## 9. `if __name__ == "__main__"` — o porteiro

Toda vez que Python carrega um arquivo, define a variável especial `__name__`:

| Como o arquivo foi carregado | Valor de `__name__` |
|------------------------------|---------------------|
| Executado diretamente (`python arquivo.py`) | `"__main__"` |
| Importado por outro (`import arquivo`) | `"arquivo"` |

Por isso o bloco `if __name__ == "__main__":` funciona como um porteiro: **o que está dentro só roda quando o arquivo é o ponto de entrada.**

**Por que isso importa:** sem ele, importar seu módulo dispararia todo o script de teste/execução. Com ele, o mesmo arquivo serve como **biblioteca** (importável) e como **programa** (executável).

In [ ]:
print("Neste notebook, __name__ vale:", repr(__name__))
print("No módulo importado, vale:    ", repr(atlas_utils.__name__))

## 🔧 Prática guiada — Refatorando o relatório em funções

Vamos pegar o relatório da aula 03 e quebrá-lo em funções com responsabilidade única. Compare a legibilidade.

In [ ]:
from collections import defaultdict

PEDIDOS = [
    {"id": 1001, "cidade": "Campinas",       "produto": "Notebook", "qtd": 2,  "preco": 2599.90, "status": "pago"},
    {"id": 1002, "cidade": "São Paulo",      "produto": "Mouse",    "qtd": 10, "preco": 89.90,   "status": "pago"},
    {"id": 1003, "cidade": "Campinas",       "produto": "Teclado",  "qtd": 3,  "preco": 249.00,  "status": "cancelado"},
    {"id": 1004, "cidade": "Sorocaba",       "produto": "Monitor",  "qtd": 1,  "preco": 1199.00, "status": "pago"},
    {"id": 1005, "cidade": "São Paulo",      "produto": "Notebook", "qtd": 1,  "preco": 2599.90, "status": "pago"},
    {"id": 1006, "cidade": "Campinas",       "produto": "Monitor",  "qtd": 4,  "preco": 1199.00, "status": "pago"},
    {"id": 1007, "cidade": "Ribeirão Preto", "produto": "Mouse",    "qtd": 25, "preco": 89.90,   "status": "pago"},
    {"id": 1008, "cidade": "São Paulo",      "produto": "Teclado",  "qtd": 6,  "preco": 249.00,  "status": "pendente"},
]


def filtrar_faturados(pedidos: list[dict]) -> list[dict]:
    """Mantém apenas os pedidos com status 'pago'."""
    return [p for p in pedidos if p["status"] == "pago"]


def valor_pedido(pedido: dict) -> float:
    """Valor bruto de um pedido."""
    return pedido["qtd"] * pedido["preco"]


def agrupar_por(pedidos: list[dict], campo: str) -> dict[str, float]:
    """Soma o valor dos pedidos agrupando por um campo qualquer.

    Repare: a função não sabe nada sobre 'cidade'. Ela agrupa por
    QUALQUER campo — é isso que a torna reutilizável.
    """
    acumulado = defaultdict(float)
    for p in pedidos:
        acumulado[p[campo]] += valor_pedido(p)
    return dict(acumulado)


def formatar_brl(valor: float) -> str:
    """Formata no padrão monetário brasileiro."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def imprimir_ranking(titulo: str, dados: dict[str, float], largura: int = 56) -> None:
    """Imprime um ranking formatado. Retorna None — é uma função de EFEITO."""
    total = sum(dados.values())
    print("=" * largura)
    print(titulo.center(largura))
    print("=" * largura)
    for pos, (chave, valor) in enumerate(sorted(dados.items(), key=lambda kv: -kv[1]), 1):
        share = valor / total if total else 0
        print(f"{pos}. {chave:<22}{formatar_brl(valor):>18}{share:>10.1%}")
    print("-" * largura)
    print(f"   {'TOTAL':<22}{formatar_brl(total):>18}")
    print()


# --- Orquestração: cada linha diz o QUE faz, não COMO ---
faturados = filtrar_faturados(PEDIDOS)

imprimir_ranking("FATURAMENTO POR CIDADE", agrupar_por(faturados, "cidade"))
imprimir_ranking("FATURAMENTO POR PRODUTO", agrupar_por(faturados, "produto"))

> 💡 **O ganho:** a regra de "o que é faturado" está em **um** lugar. Se amanhã "pendente" também contar, você muda uma linha. E `agrupar_por` já funciona para qualquer campo novo que apareça — vendedor, categoria, mês — sem escrever nada.

## 📝 Exercícios rápidos

**E1.** Escreva `eh_par(n)` que devolve `True`/`False`. Depois use-a numa comprehension para filtrar os pares de `range(1, 21)`.

**E2.** Escreva `aplicar_reajuste(precos, percentual=0.05)` que recebe uma lista e devolve uma **nova** lista reajustada, sem alterar a original. Prove que a original ficou intacta.

**E3.** Escreva `resumo(valores)` que devolve um `dict` com `total`, `media`, `minimo`, `maximo` e `qtd`. Trate o caso de lista vazia devolvendo zeros em vez de quebrar.

**E4.** Escreva `titulo_seguro(texto)` que remova espaços das pontas e aplique `.title()`, devolvendo `"(sem nome)"` se receber `None` ou string vazia.

**E5.** Crie um módulo `aurora_frete.py` com `%%writefile` contendo `calcular_frete` e um bloco `__main__` que testa 3 casos. Importe-o e chame a função.

**E6.** Explique com suas palavras (numa célula markdown) por que `def f(x, lista=[])` é perigoso e como corrigir.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

<!-- E6 — escreva sua resposta aqui -->

## ✅ Checklist de saída

- [ ] Escrevo funções com `def`, parâmetros e `return`
- [ ] Sei a diferença entre `print` e `return`
- [ ] Uso valores padrão e argumentos nomeados
- [ ] Escrevo docstring e type hints
- [ ] Entendo a ordem de busca LEGB
- [ ] Sei por que `global` deve ser evitado
- [ ] **Nunca** uso lista/dict como valor padrão de parâmetro
- [ ] Entendo por que mutar um argumento afeta o chamador
- [ ] Importo módulos das 4 formas e sei qual evitar (`import *`)
- [ ] Sei explicar `if __name__ == "__main__"`

---

### ➡️ Próxima aula

**`01_05_Arquivos_Erros_e_Debug.ipynb`** — Exceções, leitura/escrita de TXT, CSV e JSON, e depuração no VS Code. É onde os dados saem do código e passam a vir de arquivos de verdade.